> **Current notebook role — 2026-11-12.** This notebook retains the upstream stage code used by the manuscript implementation. Selected figure production uses `run.py` in `code_release/iCCA-molecular-subtyping` or the public source repository. See `docs/NOTEBOOK_STATUS.md` in that code repository for stage ownership. Earlier execution outputs are preserved in the archived notebook snapshot.

# Step 0. Data preparation, provenance, and cohort quality control

This notebook is the Step 0 record for the iCCA subtype project. It treats files in `0-Data/matrix/` as pre-QC study matrices and files in `0-Data/qc/` as the processed interfaces consumed downstream.

The notebook does four jobs: (1) inventories cohorts and assays; (2) verifies sample identifiers, dimensions, and historical RNA abundance transformations; (3) summarizes the processed expression, mutation, and proteomic resources; and (4) exports source tables, compact QC figures, a Methods-ready summary, and release-gate checks.

## 1. Context and analysis contract

### Key assumptions

- `0-Data/matrix/` contains study-level matrices and metadata before the transformations represented by Step 0.
- `0-Data/qc/` contains the canonical processed objects used by downstream notebooks.
- `0-Data/qc/sample_information.xlsx` is the curated sample-level metadata interface. Metadata embedded in historical `.Rdata` objects are checked for identifier consistency but are not used to reconstruct assay mappings.
- RNA input scale is determined from source documentation where available and checked numerically against the saved processed matrices. A constant within-sample processed/raw ratio distinguishes sample-wise TPM rescaling from unchanged TPM input.
- RNA-seq discovery cohorts and microarray projection cohorts remain separate processing and interpretation tracks.


In [ ]:
rm(list = ls())
options(stringsAsFactors = FALSE, scipen = 999)

required_packages <- c("data.table", "dplyr", "ggplot2", "patchwork", "readxl", "scales", "tidyr")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages) > 0L) {
  stop("Missing required Step 0 packages: ", paste(missing_packages, collapse = ", "))
}

suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(ggplot2)
  library(patchwork)
  library(readxl)
  library(scales)
  library(tidyr)
})

project_root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
matrix_dir <- file.path(project_root, "0-Data", "matrix")
qc_dir <- file.path(project_root, "0-Data", "qc")
figure_dir <- file.path(project_root, "0-Data", "figures")
historical_notebook <- file.path(project_root, "archive", "notebook_backups", "0-data_副本.ipynb")
dir.create(qc_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)
stopifnot(dir.exists(matrix_dir), dir.exists(qc_dir), file.exists(historical_notebook))

analysis_role_colors <- c("Discovery RNA-seq"="#3B5F8A", "Microarray projection"="#C98B3C")
study_palette_path <- file.path(qc_dir,"study_color_dictionary.csv")
stopifnot(file.exists(study_palette_path))
study_palette <- fread(study_palette_path)[order(DisplayOrder)]
stopifnot(!anyDuplicated(study_palette$Study),all(grepl("^#[0-9A-Fa-f]{6}$",study_palette$ColorHex)))
study_order <- study_palette$Study
study_colors <- setNames(study_palette$ColorHex,study_palette$Study)
canonicalize_study <- function(x) {
  x <- trimws(as.character(x))
  x[x %in% c("TCGA", "TCGA_CHOL")] <- "TCGA-CHOL"
  x[x %in% c("TIGER", "TIGERLC")] <- "TIGER-LC"
  x[x == "Deng2023"] <- "2023Deng"
  x[x == "Dong2022"] <- "2022Dong"
  x
}
normalize_sample_id <- function(x) {
  x <- trimws(as.character(x))
  x <- sub("^X(?=[0-9])", "", x, perl = TRUE)
  x <- gsub("\\.", "-", x)
  x
}
is_missing_value <- function(x) is.na(x) | trimws(as.character(x)) %in% c("", "NA", "N/A", "Unknown")
theme_step0 <- function(base_size = 9) {
  theme_classic(base_size = base_size) +
    theme(
      plot.title = element_text(face = "bold", size = base_size + 2),
      plot.subtitle = element_text(color = "#4D4D4D"),
      axis.title = element_text(face = "bold"),
      axis.text = element_text(color = "#222222"),
      strip.background = element_rect(fill = "#F2F3F4", color = NA),
      strip.text = element_text(face = "bold"),
      legend.title = element_text(face = "bold"),
      plot.margin = margin(8, 10, 8, 8)
    )
}
save_plot_pair <- function(plot, stem, width, height) {
  paths <- file.path(figure_dir, paste0(stem, c(".pdf", ".png")))
  ggsave(paths[1], plot, width = width, height = height, device = cairo_pdf, bg = "white")
  ggsave(paths[2], plot, width = width, height = height, dpi = 600, bg = "white")
  paths
}
write_qc_table <- function(x, filename) {
  path <- file.path(qc_dir, filename)
  fwrite(as.data.table(x), path)
  path
}


## 2. Canonical processed objects and cohort universe

The standalone curated sample table supplies the 1,092-sample metadata universe and preserves assay-specific identifiers such as proteomic IDs. The metadata copy embedded in the historical RNA object is retained only as an identifier-consistency check because one embedded field was simplified after preprocessing.


In [ ]:
canonical_files <- c(
  SampleInformation = file.path(qc_dir, "sample_information.xlsx"),
  RNAseq = file.path(qc_dir, "0-Data-rnaseq.Rdata"),
  MicroarrayIllumina = file.path(qc_dir, "0-Data-rnaseq-microarray1.Rdata"),
  MicroarrayAffymetrix = file.path(qc_dir, "0-Data-rnaseq-microarray2.Rdata"),
  DNAseq = file.path(qc_dir, "0-Data-dnaseq.Rdata"),
  Proteomics = file.path(qc_dir, "0-Data-protein.Rdata"),
  MutationCSV = file.path(qc_dir, "0-Data-combined_maf_data.csv"),
  CorrectedProteinCSV = file.path(qc_dir, "0-Data-corrected-protein.csv")
)
missing_canonical <- canonical_files[!file.exists(canonical_files)]
if (length(missing_canonical) > 0L) stop("Missing canonical Step 0 objects: ", paste(missing_canonical, collapse = ", "))

rna_env <- new.env(parent = emptyenv())
load(canonical_files[["RNAseq"]], envir = rna_env)
sample_info <- as.data.table(read_excel(canonical_files[["SampleInformation"]]))
sample_info[, Study := canonicalize_study(Study)]
sample_info[, SampleID := normalize_sample_id(SampleID)]
stopifnot(nrow(sample_info) == 1092L, !anyDuplicated(sample_info$SampleID))

embedded_sample_ids <- normalize_sample_id(rna_env$sample_info$SampleID)
stopifnot(setequal(sample_info$SampleID, embedded_sample_ids))
sample_info_path <- write_qc_table(sample_info, "sample_information.csv")

cohort_summary <- sample_info[, .(
  FinalSamples = .N,
  Platform = paste(sort(unique(na.omit(as.character(Platform)))), collapse = "; ")
), by = Study]
cohort_summary[, AnalysisRole := ifelse(grepl("RNAseq", Platform), "Discovery RNA-seq", "Microarray projection")]
cohort_summary[, Study := factor(Study, levels = study_order)]
setorder(cohort_summary, Study)
cohort_summary[, Study := as.character(Study)]

stopifnot(
  cohort_summary[AnalysisRole == "Discovery RNA-seq", sum(FinalSamples)] == 586L,
  cohort_summary[AnalysisRole == "Microarray projection", sum(FinalSamples)] == 506L
)
cohort_summary


## 3. Input inventory and inclusion accounting

Expression-matrix columns are counted before Step 0 selection and compared with the final processed sample universe. Exclusions here mean matrix columns not retained in the curated iCCA expression cohort; they do not automatically imply poor sample quality.


In [ ]:
source_files <- data.table(FilePath = list.files(matrix_dir, pattern = "\\.csv$", full.names = TRUE))
source_files[, File := basename(FilePath)]
source_files[, Study := canonicalize_study(sub("_(sample_info|rnaseq[0-9]*|dna|protein)\\.csv$", "", File))]
source_files[, DataType := fcase(
  grepl("_sample_info\\.csv$", File), "Sample metadata",
  grepl("_rnaseq[0-9]*\\.csv$", File), "Expression matrix",
  grepl("_dna\\.csv$", File), "Mutation matrix",
  grepl("_protein\\.csv$", File), "Protein matrix",
  default = "Other"
)]
source_files[, Rows := vapply(FilePath, function(p) nrow(fread(p, select = 1, showProgress = FALSE)), integer(1))]
source_files[, Columns := vapply(FilePath, function(p) length(strsplit(readLines(p, n = 1L, warn = FALSE), ",", fixed = TRUE)[[1]]), integer(1))]
source_files[, RelativePath := file.path("0-Data", "matrix", File)]
source_inventory <- source_files[, .(Study, DataType, File, RelativePath, Rows, Columns)]
source_inventory <- source_inventory[order(match(Study, study_order), DataType, File)]
inventory_path <- write_qc_table(source_inventory, "step0_source_file_inventory.csv")

expression_inputs <- source_files[DataType == "Expression matrix"]
expression_input_counts <- expression_inputs[, .(
  InputMatrixColumns = sum(Columns - 1L),
  SourceFiles = paste(File, collapse = "; ")
), by = Study]
cohort_inclusion <- merge(cohort_summary, expression_input_counts, by = "Study", all.x = TRUE)
cohort_inclusion[, ExcludedMatrixColumns := InputMatrixColumns - FinalSamples]
cohort_inclusion[, ExclusionDefinition := ifelse(
  ExcludedMatrixColumns > 0,
  "Matrix columns outside the curated iCCA sample universe",
  "No expression-matrix columns excluded"
)]
cohort_inclusion <- cohort_inclusion[order(match(Study, study_order))]
inclusion_path <- write_qc_table(cohort_inclusion, "step0_cohort_inclusion_manifest.csv")
cohort_inclusion


## 4. RNA abundance provenance audit

The historical notebook reused generic cells interactively, so the saved execution order alone cannot establish which studies underwent FPKM-to-TPM conversion. For each discovery cohort, this audit traces one processed sample back to its source column and compares thousands of uniquely mapped genes.

A median processed/raw ratio of 1 with zero dispersion indicates no sample-wise rescaling. A non-unit constant ratio with a processed column sum near one million identifies FPKM-to-TPM conversion.


In [ ]:
rna_objects <- c(
  "2019Ahn"="tpm_2019Ahn", "2022Dong"="tpm_2022Dong", "2023Deng"="tpm_2023Deng",
  "2023Jeon"="tpm_2023Jeon", "2023Lee"="tpm_2023Lee", "2024Fan"="tpm_2024Fan",
  "TCGA-CHOL"="tpm_tcga"
)
trace_rna_scale <- function(study, object_name) {
  processed <- rna_env[[object_name]]
  processed_sample <- colnames(processed)[1]
  source_path <- file.path(matrix_dir, paste0(study, "_rnaseq.csv"))
  header <- strsplit(readLines(source_path, n = 1L, warn = FALSE), ",", fixed = TRUE)[[1]]
  header[1] <- sub("^\\ufeff", "", header[1])
  source_ids <- header[-1]
  candidates <- processed_sample
  if (study == "2023Deng") {
    rna_id <- sample_info[Study == study & SampleID == processed_sample, as.character(RNA)][1]
    candidates <- unique(c(rna_id, gsub("_", "-", rna_id, fixed = TRUE)))
  }
  source_index <- which(source_ids %in% candidates)[1]
  if (is.na(source_index)) stop("Could not trace ", study, " sample ", processed_sample, " to the source matrix")
  source_column <- fread(
    source_path, select = c(1L, source_index + 1L), header = TRUE,
    check.names = FALSE, showProgress = FALSE, data.table = FALSE
  )
  gene <- as.character(source_column[[1]])
  raw_value <- suppressWarnings(as.numeric(source_column[[2]]))
  unique_gene <- !duplicated(gene) & !duplicated(gene, fromLast = TRUE)
  common_gene <- head(intersect(rownames(processed), gene[unique_gene]), 5000L)
  raw_match <- raw_value[match(common_gene, gene)]
  processed_match <- as.numeric(processed[common_gene, processed_sample])
  keep <- is.finite(raw_match) & is.finite(processed_match) & raw_match > 0
  ratio <- processed_match[keep] / raw_match[keep]
  ratio_median <- median(ratio)
  ratio_mad <- mad(ratio)
  rescaling <- if (abs(ratio_median - 1) < 1e-8 && ratio_mad < 1e-8) {
    "No sample-wise rescaling detected"
  } else if (abs(sum(processed[, processed_sample], na.rm = TRUE) - 1e6) < 1) {
    "Sample-wise FPKM-to-TPM rescaling detected"
  } else {
    "Unresolved"
  }
  data.table(
    Study = study,
    ProcessedSample = processed_sample,
    SourceColumn = source_ids[source_index],
    SourceColumnSum = sum(raw_value, na.rm = TRUE),
    ProcessedColumnSum = sum(processed[, processed_sample], na.rm = TRUE),
    MedianProcessedRawRatio = ratio_median,
    RatioMAD = ratio_mad,
    ComparedUniqueGenes = length(ratio),
    RescalingConclusion = rescaling
  )
}
rna_scale_audit <- rbindlist(Map(trace_rna_scale, names(rna_objects), unname(rna_objects)))
stopifnot(!any(rna_scale_audit$RescalingConclusion == "Unresolved"))
scale_audit_path <- write_qc_table(rna_scale_audit, "step0_rna_scale_audit.csv")
rna_scale_audit[, .(
  Study, SourceColumnSum = round(SourceColumnSum), ProcessedColumnSum = round(ProcessedColumnSum),
  MedianProcessedRawRatio = signif(MedianProcessedRawRatio, 5), RescalingConclusion
)]


In [ ]:
processing_contract <- data.table(
  Study = c(names(rna_objects), "2012Andersen", "2013Sia", "2017Jusakual-1", "2017Jusakual-2", "2020Job", "TIGER-LC"),
  AssayTrack = c(rep("RNA-seq discovery", 7), rep("Illumina microarray projection", 4), rep("Affymetrix microarray projection", 2)),
  SourceScale = c("TPM", "TPM", "FPKM", "TPM", "TPM", "TPM", "TPM", rep("Positive intensity", 4), rep("Normalized log2 intensity", 2)),
  SampleSelection = c(
    "Curated SampleID", "Curated SampleID", "RNA assay ID mapped to curated SampleID",
    "Curated SampleID", "Curated SampleID", "Curated SampleID", "Curated TCGA SampleID",
    rep("Curated SampleID", 6)
  ),
  DuplicateFeatureRule = c(rep("Retain duplicate gene row with highest mean abundance", 7), rep("Retain duplicate gene row with highest mean intensity", 4), rep("Median collapse by gene symbol", 2)),
  Transformation = c(
    "Filter source TPM >1 in >=20%; log2(TPM+1)",
    "Filter source TPM >1 in >=20%; log2(TPM+1)",
    "Historical source-scale >1 filter retained the same 12,666 genes as a TPM-scale sensitivity check; sample-wise FPKM-to-TPM; log2(TPM+1)",
    rep("Filter source TPM >1 in >=20%; log2(TPM+1)", 4),
    rep("log2 followed by within-cohort quantile normalization", 4),
    rep("No additional log transform or cross-platform normalization", 2)
  )
)
processing_contract_path <- write_qc_table(processing_contract, "step0_expression_processing_contract.csv")
processing_contract


## 5. Processed data dimensions and identifier integrity

All expression matrices must be genes × samples. Sample columns must be unique and recoverable from the curated metadata by identifier rather than by position.


In [ ]:
micro1_env <- new.env(parent = emptyenv())
micro2_env <- new.env(parent = emptyenv())
load(canonical_files[["MicroarrayIllumina"]], envir = micro1_env)
load(canonical_files[["MicroarrayAffymetrix"]], envir = micro2_env)

expression_objects <- list(
  "2019Ahn"=rna_env$log2tpm_2019Ahn, "2022Dong"=rna_env$log2tpm_2022Dong,
  "2023Deng"=rna_env$log2tpm_2023Deng, "2023Jeon"=rna_env$log2tpm_2023Jeon,
  "2023Lee"=rna_env$log2tpm_2023Lee, "2024Fan"=rna_env$log2tpm_2024Fan,
  "TCGA-CHOL"=rna_env$log2tpm_tcga,
  "2012Andersen"=micro1_env$log2dat_2012Andersen, "2013Sia"=micro1_env$log2dat_2013Sia,
  "2017Jusakual-1"=micro1_env$log2dat_2017Jusakual1, "2017Jusakual-2"=micro1_env$log2dat_2017Jusakual2,
  "2020Job"=micro2_env$log2dat_2020Job, "TIGER-LC"=micro2_env$log2dat_TIGER
)
expression_summary <- rbindlist(lapply(names(expression_objects), function(dataset) {
  x <- expression_objects[[dataset]]
  study <- sub("-[12]$", "", dataset)
  data.table(
    Dataset = dataset, Study = study,
    AnalysisRole = ifelse(study %in% names(rna_objects), "Discovery RNA-seq", "Microarray projection"),
    Genes = nrow(x), Samples = ncol(x),
    DuplicateGeneIDs = sum(duplicated(rownames(x))),
    DuplicateSampleIDs = sum(duplicated(colnames(x))),
    MetadataMatchedSamples = sum(normalize_sample_id(colnames(x)) %in% sample_info[Study == study, SampleID]),
    MissingValues = sum(is.na(x)), Orientation = "genes x samples"
  )
}))
stopifnot(
  all(expression_summary$DuplicateGeneIDs == 0L),
  all(expression_summary$DuplicateSampleIDs == 0L),
  all(expression_summary$MetadataMatchedSamples == expression_summary$Samples)
)
expression_summary_path <- write_qc_table(expression_summary, "step0_expression_dimensions.csv")

identifier_audit <- rbindlist(list(
  data.table(Check="Curated SampleID uniqueness", FailedN=sum(duplicated(sample_info$SampleID)), Denominator=nrow(sample_info), Note="Evaluated across the full curated cohort"),
  data.table(Check="Cross-study exact SampleID reuse", FailedN=sample_info[, uniqueN(Study), by=SampleID][, sum(V1 > 1L)], Denominator=nrow(sample_info), Note="No universal patient key exists; cryptic reuse under different internal IDs cannot be tested computationally"),
  data.table(Check="Expression sample uniqueness", FailedN=sum(expression_summary$DuplicateSampleIDs), Denominator=sum(expression_summary$Samples), Note="Evaluated within every processed expression object"),
  data.table(Check="Expression-to-metadata matching", FailedN=sum(expression_summary$Samples-expression_summary$MetadataMatchedSamples), Denominator=sum(expression_summary$Samples), Note="Matched by normalized identifiers, not column position"),
  data.table(Check="RNA input-scale classification", FailedN=sum(rna_scale_audit$RescalingConclusion=="Unresolved"), Denominator=nrow(rna_scale_audit), Note="Numerical source-to-processed tracing")
))
identifier_audit[, Status := ifelse(FailedN == 0L, "PASS", "FAIL")]
identifier_audit_path <- write_qc_table(identifier_audit, "step0_identifier_audit.csv")
expression_summary


### Feature retention from source matrices to processed interfaces

Source rows, unique source identifiers, and final processed features are kept distinct. For RNA-seq, retention reflects duplicate resolution plus the expressed-in-at-least-20% rule. For microarrays, it additionally reflects platform-specific probe/gene-symbol handling; it should not be interpreted as an RNA abundance filter.


In [ ]:
feature_file_map <- data.table(
  Dataset=c(
    "2019Ahn","2022Dong","2023Deng","2023Jeon","2023Lee","2024Fan","TCGA-CHOL",
    "2012Andersen","2013Sia","2017Jusakual-1","2017Jusakual-2","2020Job","TIGER-LC"
  ),
  File=c(
    "2019Ahn_rnaseq.csv","2022Dong_rnaseq.csv","2023Deng_rnaseq.csv","2023Jeon_rnaseq.csv",
    "2023Lee_rnaseq.csv","2024Fan_rnaseq.csv","TCGA-CHOL_rnaseq.csv",
    "2012Andersen_rnaseq.csv","2013Sia_rnaseq.csv","2017Jusakual_rnaseq1.csv",
    "2017Jusakual_rnaseq2.csv","2020Job_rnaseq.csv","TIGER-LC_rnaseq.csv"
  )
)
feature_retention <- rbindlist(lapply(seq_len(nrow(feature_file_map)), function(i) {
  dataset <- feature_file_map$Dataset[[i]]
  file <- feature_file_map$File[[i]]
  source_id <- fread(file.path(matrix_dir,file), select=1L, showProgress=FALSE)[[1L]]
  source_id <- trimws(as.character(source_id))
  source_id <- source_id[!is.na(source_id) & nzchar(source_id)]
  final_row <- expression_summary[Dataset==dataset]
  data.table(
    Dataset=dataset, Study=final_row$Study, AnalysisRole=final_row$AnalysisRole,
    SourceFile=file.path("0-Data","matrix",file), SourceRows=length(source_id),
    UniqueSourceFeatureIDs=uniqueN(source_id), FinalProcessedFeatures=final_row$Genes
  )
}))
feature_retention[, RetainedPercentOfUniqueSource := 100*FinalProcessedFeatures/UniqueSourceFeatureIDs]
feature_retention[, Interpretation := ifelse(
  AnalysisRole=="Discovery RNA-seq",
  "Duplicate resolution plus expressed-in-at-least-20-percent filter",
  "Platform-specific probe/gene-symbol handling and normalization"
)]
stopifnot(
  nrow(feature_retention)==length(expression_objects),
  all(feature_retention$FinalProcessedFeatures>0L),
  all(feature_retention$RetainedPercentOfUniqueSource>0 & feature_retention$RetainedPercentOfUniqueSource<=100)
)
feature_retention_path <- write_qc_table(feature_retention,"step0_feature_retention.csv")
expression_overview <- merge(
  expression_summary[, .(Dataset,Study,AnalysisRole,Samples,Genes)],
  feature_retention[, .(
    Dataset,SourceRows,UniqueSourceFeatureIDs,FinalProcessedFeatures,
    RetainedPercentOfUniqueSource
  )],
  by="Dataset",all=TRUE,sort=FALSE
)
expression_overview <- expression_overview[match(names(expression_objects),Dataset)]
expression_overview_path <- write_qc_table(expression_overview,"step0_expression_overview.csv")
feature_retention


## 6. Metadata definitions and completeness

Missingness is reported by study and field. Missing values are not imputed in Step 0 clinical metadata. Fields with ambiguous source definitions remain descriptive and must not be promoted into adjusted models until their definitions are harmonized.


In [ ]:
variable_definitions <- data.table(
  Field = c("SampleID", "Study", "Platform", "Sex", "Age", "Stage", "T", "N", "M",
            "Vital state (1=Dead)", "Overall survival (days)", "HBV infection (1=Positive)",
            "Liver fluke (1=Positive)", "iCCA classification", "Genome sequencing", "Protein profiling"),
  Definition = c(
    "Curated unique sample identifier", "Source study", "Expression platform", "Reported biological sex",
    "Age as supplied by the source study", "Reported pathological/clinical stage", "Tumor category", "Nodal category", "Metastasis category",
    "Overall-survival event indicator: 1=dead", "Overall-survival follow-up in days", "HBV status indicator where available",
    "Liver-fluke status indicator where available", "Observed duct/pathology classification where available",
    "Source DNA/profiling identifier", "Source proteomic identifier"
  ),
  UnitOrAllowedValues = c(
    "string", "controlled study vocabulary", "RNAseq/Microarray platform", "source categories",
    "years or source age category; inspect study definition", "source AJCC/stage vocabulary", "T category", "N category", "M category",
    "0/1", "days", "0/1 or source category", "0/1 or source category", "observed source category",
    "assay identifier", "assay identifier"
  ),
  Step0Handling = c(rep("Retained without imputation", 16))
)
variable_definitions <- variable_definitions[Field %in% names(sample_info)]
variable_definition_path <- write_qc_table(variable_definitions, "step0_metadata_variable_dictionary.csv")

metadata_missingness <- rbindlist(lapply(variable_definitions$Field, function(field) {
  sample_info[, .(
    TotalN=.N,
    MissingN=sum(is_missing_value(get(field))),
    MissingPercent=100*mean(is_missing_value(get(field)))
  ), by=Study][, Field:=field]
}))
setcolorder(metadata_missingness, c("Study", "Field", "TotalN", "MissingN", "MissingPercent"))
metadata_missingness_path <- write_qc_table(metadata_missingness, "step0_metadata_missingness.csv")

impossible_values <- rbindlist(list(
  data.table(Check="Negative overall-survival days", FailedN=if ("Overall survival (days)" %in% names(sample_info)) sum(suppressWarnings(as.numeric(sample_info[["Overall survival (days)"]])) < 0, na.rm=TRUE) else NA_integer_),
  data.table(Check="Vital-state values outside 0/1", FailedN=if ("Vital state (1=Dead)" %in% names(sample_info)) sum(!is_missing_value(sample_info[["Vital state (1=Dead)"]]) & !as.character(sample_info[["Vital state (1=Dead)"]]) %in% c("0","1")) else NA_integer_)
))
impossible_values[, Status:=ifelse(is.na(FailedN), "NOT_APPLICABLE", ifelse(FailedN==0,"PASS","FAIL"))]
impossible_values_path <- write_qc_table(impossible_values, "step0_metadata_validity_checks.csv")
metadata_missingness[order(-MissingPercent)][1:12]


## 7. Assay-specific processed resources

Mutation and proteomic counts are reported at their available assay grain. They are not generalized to the full 1,092-sample metadata universe.


In [ ]:
dna_env <- new.env(parent = emptyenv())
protein_env <- new.env(parent = emptyenv())
load(canonical_files[["DNAseq"]], envir = dna_env)
load(canonical_files[["Proteomics"]], envir = protein_env)
maf <- as.data.table(dna_env$combined_maf_data)
maf[, Study := canonicalize_study(Dataset)]

protein_matrix <- protein_env$corrected_prot_matrix
protein_samples <- normalize_sample_id(colnames(protein_matrix))
dong_protein_ids <- normalize_sample_id(setdiff(names(protein_env$pro_2022Dong), "Gene"))
protein_assignment <- data.table(
  SampleID=protein_samples,
  Study=ifelse(protein_samples %in% dong_protein_ids, "2022Dong", "2023Deng")
)
protein_summary <- protein_assignment[, .(
  Samples=.N,
  MatchedCuratedExpressionSamples=sum(SampleID %in% sample_info[Study == .BY$Study, SampleID])
), by=Study][, `:=`(
  Assay="Protein (mass spectrometry)",
  DenominatorDefinition="Columns in the Dong/Deng batch-corrected mass-spectrometry matrix"
)]

tcga_rppa <- as.data.table(protein_env$pro_tcga)
tcga_rppa_annotation_columns <- intersect(
  c("AGID", "lab_id", "catalog_number", "set_id", "peptide_target"),
  names(tcga_rppa)
)
tcga_rppa_sample_columns <- setdiff(names(tcga_rppa), tcga_rppa_annotation_columns)
tcga_rppa_sample_ids <- normalize_sample_id(tcga_rppa_sample_columns)
tcga_rppa_usable_targets <- sum(
  rowSums(!is.na(as.matrix(tcga_rppa[, ..tcga_rppa_sample_columns]))) > 0L
)
tcga_rppa_summary <- data.table(
  Study="TCGA-CHOL",
  Assay="Protein (RPPA)",
  Samples=length(tcga_rppa_sample_ids),
  MatchedCuratedExpressionSamples=sum(
    tcga_rppa_sample_ids %in% sample_info[Study == "TCGA-CHOL", SampleID]
  ),
  DenominatorDefinition=paste0(
    "RPPA sample columns; ", tcga_rppa_usable_targets,
    " targets with at least one observed value"
  )
)

dna_summary <- maf[, .(AssaySampleID=unique(normalize_sample_id(Tumor_Sample_Barcode))), by=Study]
dna_summary[, Matched := if (.BY$Study == "TCGA-CHOL") {
  any(startsWith(sample_info[Study == .BY$Study, SampleID], AssaySampleID))
} else {
  AssaySampleID %in% sample_info[Study == .BY$Study, SampleID]
}, by=.(Study, AssaySampleID)]
dna_summary <- dna_summary[, .(
  Samples=.N,
  MatchedCuratedExpressionSamples=sum(Matched)
), by=Study][, `:=`(Assay="DNA-seq", DenominatorDefinition="Unique samples represented in harmonized mutation records")]

assay_summary <- rbindlist(list(
  cohort_summary[, .(Study, Assay="Expression", Samples=FinalSamples, MatchedCuratedExpressionSamples=FinalSamples, DenominatorDefinition="Curated expression cohort")],
  dna_summary,
  protein_summary,
  tcga_rppa_summary
), fill=TRUE)
assay_summary <- assay_summary[order(match(Study, study_order), Assay)]
assay_summary_path <- write_qc_table(assay_summary, "step0_assay_denominators.csv")

object_summary <- data.table(
  Object=c("Harmonized mutation table", "Batch-corrected mass-spectrometry matrix", "TCGA-CHOL RPPA matrix"),
  Rows=c(nrow(maf), nrow(protein_matrix), nrow(tcga_rppa)),
  Columns=c(ncol(maf), ncol(protein_matrix), length(tcga_rppa_sample_columns)),
  UnitOfAnalysis=c("variant record", "protein x sample", "RPPA target x sample"),
  CanonicalPath=c(
    "0-Data/qc/0-Data-dnaseq.Rdata",
    "0-Data/qc/0-Data-protein.Rdata",
    "0-Data/qc/0-Data-protein.Rdata"
  )
)
object_summary_path <- write_qc_table(object_summary, "step0_multiomics_object_summary.csv")
assay_summary


### Assay availability and protein batch diagnostics

Assay availability distinguishes total assay samples from the subset matched to the curated expression universe. Protein batch diagnostics reconstruct the pre-ComBat matrix from protected source files, verify the recomputed post-ComBat matrix against the locked canonical object, and project both stages into the same pre-ComBat PCA basis.


In [ ]:
assay_availability <- merge(
  CJ(
    Study=study_order,
    Assay=c("Expression","DNA-seq","Protein (mass spectrometry)","Protein (RPPA)"),
    unique=TRUE
  ),
  assay_summary,
  by=c("Study","Assay"),all.x=TRUE,sort=FALSE
)
assay_availability[, Available := !is.na(Samples) & Samples>0L]
assay_availability[, CoveragePercent := fifelse(
  Available,100*MatchedCuratedExpressionSamples/Samples,NA_real_
)]
assay_availability[, DisplayLabel := fifelse(
  Available,paste0(MatchedCuratedExpressionSamples,"/",Samples),"—"
)]
assay_availability <- assay_availability[
  order(match(Study,study_order),match(
    Assay,c("Expression","DNA-seq","Protein (mass spectrometry)","Protein (RPPA)")
  ))
]
assay_availability_path <- write_qc_table(assay_availability,"step0_assay_availability_matrix.csv")

protein_qc_script <- file.path(project_root,"0-Data","scripts","generate_step0_supplementary_qc.R")
stopifnot(file.exists(protein_qc_script))
protein_qc_log <- capture.output(
  suppressMessages(sys.source(protein_qc_script,envir=new.env(parent=globalenv()))),
  type="output"
)
protein_pca_scores_path <- file.path(qc_dir,"step0_protein_pca_scores.csv")
protein_pca_metrics_path <- file.path(qc_dir,"step0_protein_batch_qc_metrics.csv")
protein_pca_features_path <- file.path(qc_dir,"step0_protein_pca_feature_selection.csv")
stopifnot(all(file.exists(c(protein_pca_scores_path,protein_pca_metrics_path,protein_pca_features_path))))
protein_pca_scores <- fread(protein_pca_scores_path)
protein_pca_metrics <- fread(protein_pca_metrics_path)
stopifnot(
  nrow(protein_pca_scores)==596L,
  setequal(protein_pca_scores$Stage,c("Pre-ComBat","Post-ComBat")),
  all(protein_pca_metrics$RecomputedCanonicalMaxAbsDifference<1e-10)
)
list(AssayAvailability=assay_availability,ProteinBatchMetrics=protein_pca_metrics)


## 8. Compact publication-facing QC figures

These plots are candidate supplementary QC artifacts. They are generated from saved source tables and avoid assigning final manuscript numbers. Study identity is carried by direct labels wherever possible; color encodes analysis track unless study is the comparison itself. RNA-seq batch-correction PCA is deliberately deferred to Step1 because the corrected cross-study expression object belongs to that step's release gate; microarray platforms remain separate projection tracks rather than being force-corrected together.


In [ ]:
cohort_plot_data <- copy(cohort_inclusion)
cohort_plot_data[, Study := factor(Study, levels=rev(study_order))]
p_cohort <- ggplot(cohort_plot_data, aes(x=FinalSamples, y=Study, fill=AnalysisRole)) +
  geom_col(width=0.72) +
  geom_text(aes(label=FinalSamples), hjust=-0.15, size=2.8) +
  facet_grid(AnalysisRole ~ ., scales="free_y", space="free_y") +
  scale_fill_manual(values=analysis_role_colors, guide="none", drop=FALSE) +
  scale_x_continuous(expand=expansion(mult=c(0,0.12))) +
  labs(title="Expression cohorts comprise 586 discovery and 506 projection samples",
       subtitle="Final curated samples after study-specific identifier matching", x="Samples", y=NULL) +
  theme_step0(9)
cohort_figure_paths <- save_plot_pair(p_cohort, "step0_cohort_composition", 8.2, 6.3)
p_cohort


In [ ]:
sample_plot_data <- copy(expression_summary)
sample_plot_data[,Dataset:=factor(Dataset,levels=rev(expression_summary$Dataset))]
p_samples <- ggplot(sample_plot_data, aes(x=Samples, y=Dataset, fill=AnalysisRole)) +
  geom_col(width=0.7) +
  geom_text(aes(label=comma(Samples)), hjust=-0.1, size=2.5) +
  scale_fill_manual(values=c("Discovery RNA-seq"="#3366A5", "Microarray projection"="#D98B3A")) +
  scale_x_continuous(expand=expansion(mult=c(0,0.2))) +
  labs(title="A. Curated expression samples by study",
       subtitle="All sample identifiers match the curated metadata interface",
       x="Samples", y=NULL, fill=NULL) +
  theme_step0(8.5) + theme(legend.position="top")
p_samples


In [ ]:
heatmap_fields <- intersect(c("Sex", "Age", "Stage", "Vital state (1=Dead)", "Overall survival (days)",
                              "HBV infection (1=Positive)", "Liver fluke (1=Positive)", "iCCA classification",
                              "Genome sequencing", "Protein profiling"), variable_definitions$Field)
missing_plot_data <- metadata_missingness[Field %in% heatmap_fields]
missing_plot_data[, Study:=factor(Study, levels=rev(study_order))]
missing_plot_data[, Field:=factor(Field, levels=heatmap_fields)]
p_missing <- ggplot(missing_plot_data, aes(x=Field, y=Study, fill=100-MissingPercent)) +
  geom_tile(color="white", linewidth=0.3) +
  geom_text(aes(label=sprintf("%.0f", 100-MissingPercent)), size=2.1) +
  scale_fill_gradient(low="#F7FBFF", high="#2166AC", limits=c(0,100), name="Complete (%)") +
  labs(title="Clinical metadata completeness is study- and field-specific",
       subtitle="Numbers show the percentage of curated samples with non-missing values", x=NULL, y=NULL) +
  theme_step0(8) +
  theme(axis.text.x=element_text(angle=40, hjust=1), legend.position="right")
missing_figure_paths <- save_plot_pair(p_missing, "step0_metadata_completeness", 10.5, 5.8)
p_missing


In [ ]:
inclusion_flow_data <- rbindlist(list(
  cohort_inclusion[,.(Study,AnalysisRole,Status="Retained",Samples=FinalSamples)],
  cohort_inclusion[,.(Study,AnalysisRole,Status="Excluded",Samples=pmax(ExcludedMatrixColumns,0L))]
))
inclusion_flow_data[,Study:=factor(Study,levels=rev(study_order))]
inclusion_label_data <- copy(cohort_inclusion)
inclusion_label_data[,Study:=factor(Study,levels=rev(study_order))]
p_inclusion <- ggplot(inclusion_flow_data,aes(x=Samples,y=Study,fill=Status))+
  geom_col(width=0.72,color="#FFFFFF",linewidth=0.2)+
  geom_text(
    data=inclusion_label_data,
    aes(x=InputMatrixColumns,y=Study,label=paste0(FinalSamples,"/",InputMatrixColumns)),
    inherit.aes=FALSE,hjust=-0.12,size=2.7,color="#222222"
  )+
  facet_grid(AnalysisRole~.,scales="free_y",space="free_y")+
  scale_fill_manual(values=c("Retained"="#3366A5","Excluded"="#E5A35A"))+
  scale_x_continuous(expand=expansion(mult=c(0,0.18)))+
  labs(
    title="Expression-matrix sample inclusion by study",
    subtitle="Labels show retained/input matrix columns; exclusions denote columns outside the curated iCCA universe",
    x="Input matrix columns",y=NULL,fill=NULL
  )+
  theme_step0(9)+theme(legend.position="top")
inclusion_flow_paths <- save_plot_pair(p_inclusion,"step0_sample_inclusion_flow",8.7,6.3)
p_inclusion


In [ ]:
assay_plot_data <- copy(assay_availability)
assay_plot_data[,Study:=factor(Study,levels=rev(study_order))]
assay_plot_data[,Assay:=factor(
  Assay,
  levels=c("Expression","DNA-seq","Protein (mass spectrometry)","Protein (RPPA)")
)]
assay_plot_data[,LabelColor:=ifelse(!is.na(CoveragePercent)&CoveragePercent>=75,"white","#1F2933")]
p_assay <- ggplot(assay_plot_data,aes(x=Assay,y=Study,fill=CoveragePercent))+
  geom_tile(color="white",linewidth=0.7)+
  geom_text(aes(label=DisplayLabel,color=LabelColor),size=2.7)+
  scale_fill_gradient(low="#DCEAF7",high="#3366A5",limits=c(0,100),na.value="#F1F2F3",name="Matched (%)")+
  scale_color_identity()+
  labs(
    title="Assay availability and expression-matched denominators",
    subtitle="Cell labels are matched curated-expression samples / all available assay samples",
    x=NULL,y=NULL
  )+
  theme_step0(9)+theme(panel.grid=element_blank())
assay_availability_paths <- save_plot_pair(p_assay,"step0_assay_availability",9.2,5.8)
p_assay


In [ ]:
feature_plot_data <- copy(feature_retention)
feature_plot_data[,Dataset:=factor(Dataset,levels=rev(expression_summary$Dataset))]
p_feature_retention <- ggplot(feature_plot_data,aes(x=RetainedPercentOfUniqueSource,y=Dataset,fill=AnalysisRole))+
  geom_col(width=0.68)+
  geom_text(
    aes(label=paste0(comma(FinalProcessedFeatures)," / ",comma(UniqueSourceFeatureIDs))),
    hjust=-0.08,size=2.45
  )+
  scale_fill_manual(values=c("Discovery RNA-seq"="#3366A5","Microarray projection"="#D98B3A"))+
  scale_x_continuous(limits=c(0,110),breaks=seq(0,100,20),labels=function(x)paste0(x,"%"))+
  labs(
    title="B. Feature retention from source matrices to processed interfaces",
    subtitle="Labels show final processed features / unique source feature identifiers",
    x="Retained features (%)",y=NULL,fill=NULL
  )+
  theme_step0(8.5)+theme(legend.position="none")
p_expression_overview <- p_samples / p_feature_retention +
  plot_layout(heights=c(1,1.25)) +
  plot_annotation(
    title="Expression-resource overview",
    subtitle="Curated sample counts and study-specific feature retention are shown on aligned study axes"
  )
expression_overview_paths <- save_plot_pair(p_expression_overview,"step0_expression_overview",9.2,10.6)
p_expression_overview


In [ ]:
protein_plot_data <- copy(protein_pca_scores)
protein_plot_data[,Stage:=factor(Stage,levels=c("Pre-ComBat","Post-ComBat"))]
protein_plot_data[,MatchedStatus:=ifelse(MatchedCuratedExpression,"Expression-matched","Assay-only")]
pre_eta <- protein_pca_metrics[Stage=="Pre-ComBat",WeightedStudyEtaSquaredTop10]
post_eta <- protein_pca_metrics[Stage=="Post-ComBat",WeightedStudyEtaSquaredTop10]
p_protein_pca <- ggplot(protein_plot_data,aes(x=PC1,y=PC2,color=Study,shape=MatchedStatus))+
  geom_point(size=1.8,alpha=0.78,stroke=0.55)+
  facet_wrap(~Stage,nrow=1)+
  scale_color_manual(values=study_colors[c("2022Dong","2023Deng")])+
  scale_shape_manual(values=c("Expression-matched"=16,"Assay-only"=1))+
  labs(
    title="Protein PCA before and after ComBat",
    subtitle=sprintf("Shared standardized pre-ComBat PCA basis (2,000 variable proteins); weighted study eta-squared across PC1-PC10: %.3f to %.4f",pre_eta,post_eta),
    x="PC1 (shared pre-ComBat basis)",y="PC2 (shared pre-ComBat basis)",color="Study",shape=NULL
  )+
  theme_step0(9)+theme(legend.position="top")
protein_pca_paths <- save_plot_pair(p_protein_pca,"step0_protein_pca_combat",10.2,4.8)
p_protein_pca


## 9. Methods-ready summary and release-gate evidence

The text below is generated from the locked processing contract and observed cohort counts. It is a source paragraph for manuscript synchronization, not a substitute for the final edited Methods section.


In [ ]:
methods_lines <- c(
  "Step 0 preprocessing summary",
  "",
  paste0("The curated expression resource contained 1,092 iCCA samples: 586 RNA-seq samples from seven discovery cohorts and 506 samples from five microarray projection cohorts."),
  "Study-level matrices and metadata were read from 0-Data/matrix, whereas processed downstream interfaces were stored in 0-Data/qc.",
  "Curated sample metadata were loaded from the standalone Step 0 sample-information interface; the 1,092 sample identifiers agreed with the metadata copy embedded in the historical RNA object. Assay-specific identifiers were retained only in the standalone interface.",
  "Within each expression dataset, duplicate gene identifiers were resolved using the documented platform-specific rule, and sample columns were matched to curated identifiers before integration.",
  "For six RNA-seq cohorts, numerical tracing showed that the processed TPM values were unchanged from the source TPM scale before log2(TPM+1) transformation. For 2023Deng, a constant within-sample processed/raw ratio and a processed column sum of one million confirmed sample-wise FPKM-to-TPM conversion.",
  "RNA-seq genes were retained when expression exceeded 1 in at least 20% of the study-specific samples. In 2023Deng, the historical source-scale filter and a TPM-scale sensitivity check retained the same 12,666 genes.",
  "Illumina microarray cohorts were log2 transformed and quantile normalized within cohort. Affymetrix inputs were treated as normalized log2 intensities and duplicate gene mappings were collapsed by the median without cross-platform normalization.",
  "RNA-seq discovery and microarray projection data were maintained as separate processing tracks. Clinical metadata were not imputed, and assay-specific denominators were retained for mutation and protein analyses.",
  sprintf("For the 298-sample Dong/Deng mass-spectrometry matrix, PCA projection in a shared feature-standardized pre-ComBat basis showed that the variance-weighted study eta-squared across PC1-PC10 decreased from %.3f before ComBat to %.4f after ComBat; the independently recomputed corrected matrix matched the canonical qc object.",pre_eta,post_eta),
  paste0("TCGA-CHOL RPPA was retained as a separate platform: ",length(tcga_rppa_sample_ids)," RPPA samples were available, ",sum(tcga_rppa_sample_ids %in% sample_info[Study == "TCGA-CHOL", SampleID])," matched the curated expression cohort, and ",tcga_rppa_usable_targets," targets contained at least one observed value. RPPA values were not merged with or ComBat-corrected together with mass-spectrometry abundance."),
  "No exact SampleID was reused across studies. Because the source metadata do not contain a universal cross-study patient identifier, reuse under unrelated internal identifiers cannot be excluded computationally and remains a source-provenance limitation."
)
methods_summary_path <- file.path(qc_dir, "step0_methods_preprocessing_summary.md")
writeLines(methods_lines, methods_summary_path)
cat(paste(methods_lines, collapse="\n"))


In [ ]:
session_path <- file.path(qc_dir, "step0_session_info.txt")
writeLines(capture.output(sessionInfo()), session_path)

rebuild_comparison_path <- file.path(qc_dir, "step0_rebuild_comparison.csv")
if (!file.exists(rebuild_comparison_path)) stop("Run the isolated Step 0 rebuild comparison before release-gate evaluation")
rebuild_comparison <- fread(rebuild_comparison_path)

gate_checks <- data.table(
  Check=c(
    "Canonical qc interfaces exist", "Discovery sample count is 586", "Projection sample count is 506",
    "Curated SampleID is unique", "All expression samples match metadata",
    "RNA abundance rescaling is resolved", "Expression matrices are genes x samples",
    "Metadata validity checks have no failures", "No exact cross-study SampleID reuse is present",
    "Protein ComBat diagnostics reproduce the canonical object and reduce study association",
    "TCGA-CHOL RPPA availability and matched denominator are explicit",
    "Isolated matrix-to-qc rebuild reproduces analytical values",
    "Required Step 0 source tables are non-empty"
  ),
  Passed=c(
    all(file.exists(canonical_files)),
    cohort_summary[AnalysisRole=="Discovery RNA-seq",sum(FinalSamples)]==586L,
    cohort_summary[AnalysisRole=="Microarray projection",sum(FinalSamples)]==506L,
    !anyDuplicated(sample_info$SampleID),
    all(expression_summary$MetadataMatchedSamples==expression_summary$Samples),
    !any(rna_scale_audit$RescalingConclusion=="Unresolved"),
    all(expression_summary$Orientation=="genes x samples"),
    !any(impossible_values$Status=="FAIL"),
    identifier_audit[Check=="Cross-study exact SampleID reuse",FailedN]==0L,
    all(protein_pca_metrics$RecomputedCanonicalMaxAbsDifference<1e-10) && pre_eta>post_eta,
    assay_summary[Study=="TCGA-CHOL" & Assay=="Protein (RPPA)", Samples]==30L &&
      assay_summary[Study=="TCGA-CHOL" & Assay=="Protein (RPPA)", MatchedCuratedExpressionSamples]==25L,
    nrow(rebuild_comparison)>0L && all(rebuild_comparison$Status=="PASS"),
    all(vapply(c(inventory_path,inclusion_path,scale_audit_path,processing_contract_path,
                 expression_summary_path,expression_overview_path,identifier_audit_path,variable_definition_path,
                 metadata_missingness_path,assay_summary_path,assay_availability_path,
                 feature_retention_path,study_palette_path,object_summary_path,protein_pca_scores_path,
                 protein_pca_metrics_path,protein_pca_features_path),
               function(p) file.exists(p)&&file.info(p)$size>0, logical(1)))
  )
)
gate_checks[, Evidence := c(
  paste(names(canonical_files), collapse="; "), "step0_cohort_inclusion_manifest.csv",
  "step0_cohort_inclusion_manifest.csv", "step0_identifier_audit.csv",
  "step0_identifier_audit.csv", "step0_rna_scale_audit.csv",
  "step0_expression_dimensions.csv", "step0_metadata_validity_checks.csv", "step0_identifier_audit.csv",
  "step0_protein_batch_qc_metrics.csv; step0_protein_pca_scores.csv",
  "step0_assay_denominators.csv; step0_assay_availability_matrix.csv; 0-Data/qc/0-Data-protein.Rdata",
  "step0_rebuild_comparison.csv; 0-Data/scripts/rebuild_step0_processed_objects.R; 0-Data/scripts/compare_step0_rebuild.R",
  "Step 0 QC source tables"
)]
gate_checks[, Status:=ifelse(Passed,"PASS","FAIL")]
gate_path <- write_qc_table(gate_checks, "step0_release_gate_checks.csv")
gate_checks


In [ ]:
supplementary_rows <- rbindlist(list(
  data.table(
    Type="table",
    Label=c(
      "Step 0 source-file inventory",
      "Step 0 cohort inclusion manifest",
      "Step 0 expression-resource overview source",
      "Step 0 metadata missingness",
      "Step 0 assay availability including separate TCGA-CHOL RPPA"
    ),
    Path=c(
      inventory_path,inclusion_path,expression_overview_path,
      metadata_missingness_path,assay_availability_path
    )
  ),
  data.table(
    Type="figure",
    Label=rep(c(
      "Cohort composition","Expression-resource overview","Metadata completeness",
      "Sample inclusion","Assay availability including separate TCGA-CHOL RPPA",
      "Mass-spectrometry pre/post-ComBat PCA"
    ),each=2L),
    Path=c(
      cohort_figure_paths,expression_overview_paths,missing_figure_paths,
      inclusion_flow_paths,assay_availability_paths,protein_pca_paths
    )
  )
))
supplementary_rows[,Path:=sub(
  paste0("^",project_root,"/"),"",normalizePath(Path,winslash="/",mustWork=FALSE)
)]
supplementary_rows[,Exists:=file.exists(file.path(project_root,Path))]
supplementary_rows[,NonEmpty:=Exists & file.info(file.path(project_root,Path))$size>0]
supplementary_manifest_path <- write_qc_table(
  supplementary_rows,"step0_supplementary_manifest.csv"
)

artifact_rows <- rbindlist(list(
  data.table(ArtifactType="source_table", ArtifactRole="Step 0 evidence", Path=c(
    inventory_path,inclusion_path,scale_audit_path,processing_contract_path,expression_summary_path,study_palette_path,
    identifier_audit_path,variable_definition_path,metadata_missingness_path,impossible_values_path,
    assay_summary_path,assay_availability_path,feature_retention_path,expression_overview_path,object_summary_path,
    protein_pca_scores_path,protein_pca_metrics_path,protein_pca_features_path,
    rebuild_comparison_path,gate_path,sample_info_path,supplementary_manifest_path
  )),
  data.table(ArtifactType="figure", ArtifactRole="Candidate supplementary QC figure", Path=c(
    cohort_figure_paths,expression_overview_paths,missing_figure_paths,inclusion_flow_paths,
    assay_availability_paths,protein_pca_paths
  )),
  data.table(ArtifactType="analysis_script", ArtifactRole="Protein batch diagnostic reconstruction", Path=protein_qc_script),
  data.table(ArtifactType="methods_summary", ArtifactRole="Methods evidence", Path=methods_summary_path),
  data.table(ArtifactType="environment", ArtifactRole="Reproducibility evidence", Path=session_path)
), fill=TRUE)
artifact_rows[, Path:=sub(paste0("^",project_root,"/"),"",normalizePath(Path,winslash="/",mustWork=FALSE))]
artifact_rows[, Exists:=file.exists(file.path(project_root,Path))]
artifact_rows[, NonEmpty:=Exists & file.info(file.path(project_root,Path))$size>0]
artifact_rows[, ProvisionalManuscriptNumber:=NA_character_]
output_manifest_path <- write_qc_table(artifact_rows, "step0_output_manifest.csv")
stopifnot(all(artifact_rows$Exists),all(artifact_rows$NonEmpty),all(gate_checks$Passed))
artifact_rows


## 10. Takeaways

- The expression cohort contains 586 discovery RNA-seq samples and 506 microarray projection samples, with no duplicate curated SampleID.
- Numerical tracing supports unchanged source TPM for six discovery studies and sample-wise FPKM-to-TPM conversion for 2023Deng.
- Processed expression matrices are genes × samples, have unique row and column identifiers, and match the curated metadata by sample identifier.
- Assay coverage is reported as matched/available rather than a single mixed denominator. The mass-spectrometry matrix contains 214 2022Dong and 84 2023Deng assays; TCGA-CHOL RPPA remains a separate 30-sample platform with 25 expression-matched tumors and 456 usable targets.
- In the shared standardized protein PCA basis, variance-weighted study eta-squared across PC1-PC10 decreased from 0.774 before ComBat to 0.0005 after ComBat, while the recomputed corrected matrix matched the canonical object.
- Metadata completeness varies substantially by study and variable; downstream analyses must report analysis-specific denominators and must not silently treat missing clinical data as complete.
- RNA-seq batch-correction PCA remains a Step1 deliverable, and microarray projection cohorts are not combined into a misleading cross-platform correction plot.
- Final supplementary numbering remains deferred until the manuscript-wide figure and table assembly step.
